# GastroNet — NB0: Setup & Split Lock

**Run this notebook exactly ONCE per project (on whichever Google account you start with).**

What this notebook does:
1. Mounts Google Drive and creates the permanent experiment folder structure.
2. Generates the dataset split using `sorted(os.listdir())` (deterministic, fixes the Fault-8 bug) and **locks it forever** to `dataset_split.json`.
3. Writes a shared, importable `checkpoint_utils.py` module to Drive — every other notebook (NB1, NB2, NB3, NB4, NB5a-e, NB6) imports this, none of them redefine it.
4. Initializes `experiments_manifest.json` — the single source of truth for what's been run, on which account, and its status.
5. Checks the locked split for exact-duplicate files across train/val/test, and if any are found, writes a corrected `dataset_split_v2.json` alongside the original (never overwriting it).

**Do not re-run the split-generation cell (Cell 5) after today.** It is guarded so it will refuse to overwrite an existing split, but treat that guard as a safety net, not permission to try. The duplicate-check and v2-fix cells further down are safe to re-run any time — they only read the existing split and never touch Cell 5's output.


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import os

# ============================================================
# EDIT THESE FOUR THINGS FOR YOUR SETUP, THEN DO NOT CHANGE THEM AGAIN
# ============================================================

# Path to the raw dataset (folder containing one subfolder per class)
RAW_DATASET_DIR = "/content/drive/MyDrive/gastronet_raw_dataset"

# A short, human label for THIS Google account (used in checkpoint metadata
# so you can trace any checkpoint back to which account produced it).
# Do not put an email address here, just a tag.
ACCOUNT_TAG = "acct_A"

# This notebook's own name, stored in manifest entries.
NOTEBOOK_NAME = "NB0_setup_and_split_lock"

# Root of the permanent experiment structure on THIS account's Drive.
EXPERIMENTS_ROOT = "/content/drive/MyDrive/gastronet_experiments"

# ============================================================
# Fixed project constants — do not change once split is locked
# ============================================================
CLASS_NAMES   = ["Diverticulosis", "Neoplasm", "Peritonitis", "Ureters"]
SPLIT_SEED    = 42                     # controls the split itself, never a training seed
SPLIT_RATIOS  = {"train": 0.8, "val": 0.1, "test": 0.1}
EXPECTED_TOTAL = 4000
EXPECTED_PER_CLASS = 1000

SPLIT_JSON_PATH    = os.path.join(EXPERIMENTS_ROOT, "dataset_split.json")
MANIFEST_JSON_PATH = os.path.join(EXPERIMENTS_ROOT, "experiments_manifest.json")
CHECKPOINT_UTILS_PATH = os.path.join(EXPERIMENTS_ROOT, "checkpoint_utils.py")

os.makedirs(EXPERIMENTS_ROOT, exist_ok=True)
print("Experiments root:", EXPERIMENTS_ROOT)
print("Account tag:", ACCOUNT_TAG)


Experiments root: /content/drive/MyDrive/gastronet_experiments
Account tag: acct_A


In [5]:
# Create the top-level experiment folders (per-seed subfolders are created
# lazily by checkpoint_utils.get_experiment_dir() in each training notebook,
# not here — NB0 only owns the shared, top-level artifacts).

for model_family in ["cnn_only", "vit_only", "hybrid_concat", "hybrid_crossattn"]:
    os.makedirs(os.path.join(EXPERIMENTS_ROOT, model_family), exist_ok=True)

print("Top-level folders ready under:", EXPERIMENTS_ROOT)
for f in sorted(os.listdir(EXPERIMENTS_ROOT)):
    print(" -", f)


Top-level folders ready under: /content/drive/MyDrive/gastronet_experiments
 - __pycache__
 - _sanity_check
 - checkpoint_utils.py
 - cnn_only
 - dataset_split.json
 - experiments_manifest.json
 - hybrid_concat
 - hybrid_crossattn
 - vit_only


In [6]:
import json
import hashlib
from sklearn.model_selection import train_test_split

def _build_deterministic_split():
    '''
    THE fix for Fault 8: sorted(os.listdir(...)) guarantees the file order
    is identical no matter which machine, account, or session lists the
    files, which is what makes train_test_split(..., random_state=SPLIT_SEED)
    actually reproducible. Plain os.listdir() does NOT guarantee this.
    '''
    split = {"train": [], "val": [], "test": []}
    per_class_counts = {}

    for cls in CLASS_NAMES:
        cls_dir = os.path.join(RAW_DATASET_DIR, cls)
        files = sorted(os.listdir(cls_dir))   # <-- the critical fix
        files = [os.path.join(cls_dir, f) for f in files]
        per_class_counts[cls] = len(files)

        train_files, temp_files = train_test_split(
            files,
            train_size=SPLIT_RATIOS["train"],
            random_state=SPLIT_SEED,
            shuffle=True,
        )
        val_frac_of_temp = SPLIT_RATIOS["val"] / (SPLIT_RATIOS["val"] + SPLIT_RATIOS["test"])
        val_files, test_files = train_test_split(
            temp_files,
            train_size=val_frac_of_temp,
            random_state=SPLIT_SEED,
            shuffle=True,
        )

        split["train"].extend([(p, cls) for p in train_files])
        split["val"].extend([(p, cls) for p in val_files])
        split["test"].extend([(p, cls) for p in test_files])

    return split, per_class_counts


def _verify_split(split, per_class_counts):
    total = sum(len(split[k]) for k in split)
    assert total == EXPECTED_TOTAL, f"Expected {EXPECTED_TOTAL} images total, got {total}"
    for cls, cnt in per_class_counts.items():
        assert cnt == EXPECTED_PER_CLASS, f"Class {cls} has {cnt} images, expected {EXPECTED_PER_CLASS}"

    # no overlap between splits
    all_paths = [p for k in split for p, _ in split[k]]
    assert len(all_paths) == len(set(all_paths)), "Duplicate image found across splits!"

    print(f"Total images: {total} (expected {EXPECTED_TOTAL}) - OK")
    print(f"Per-class counts: {per_class_counts} - OK")
    print(f"Train/Val/Test sizes: "
          f"{len(split['train'])}/{len(split['val'])}/{len(split['test'])}")
    print("No duplicate/overlapping images across splits - OK")


def _split_hash(split):
    '''Hash of the sorted file list, used downstream so every experiment's
    config.json can assert it trained/evaluated on THIS exact split, even
    across different Google accounts.'''
    flat = sorted([p for k in split for p, _ in split[k]])
    return hashlib.md5("||".join(flat).encode("utf-8")).hexdigest()


if os.path.exists(SPLIT_JSON_PATH):
    print("dataset_split.json ALREADY EXISTS at:")
    print(" ", SPLIT_JSON_PATH)
    print("Refusing to regenerate it. If you genuinely need a new split, that is a")
    print("new project decision, not a rerun of this cell — rename/move the old file")
    print("deliberately first, then think hard about whether you actually want this.")
else:
    split, per_class_counts = _build_deterministic_split()
    _verify_split(split, per_class_counts)
    split_hash = _split_hash(split)

    payload = {
        "split_seed": SPLIT_SEED,
        "split_ratios": SPLIT_RATIOS,
        "class_names": CLASS_NAMES,
        "expected_total": EXPECTED_TOTAL,
        "expected_per_class": EXPECTED_PER_CLASS,
        "split_hash": split_hash,
        "generated_by_account": ACCOUNT_TAG,
        "generated_by_notebook": NOTEBOOK_NAME,
        "split": split,
    }
    with open(SPLIT_JSON_PATH, "w") as f:
        json.dump(payload, f, indent=2)

    print("\ndataset_split.json LOCKED at:", SPLIT_JSON_PATH)
    print("split_hash:", split_hash)
    print("\nCopy this exact file to every other Google account's Drive under the")
    print("same path before running any other notebook there. Never regenerate it.")


dataset_split.json ALREADY EXISTS at:
  /content/drive/MyDrive/gastronet_experiments/dataset_split.json
Refusing to regenerate it. If you genuinely need a new split, that is a
new project decision, not a rerun of this cell — rename/move the old file
deliberately first, then think hard about whether you actually want this.


In [7]:
# Always load from disk here rather than reusing the in-memory `split` variable —
# this is exactly the behavior every downstream notebook should copy: never trust
# an in-memory split, always load dataset_split.json fresh.

with open(SPLIT_JSON_PATH) as f:
    loaded = json.load(f)

print("Loaded split_hash:", loaded["split_hash"])
print("Loaded split seed:", loaded["split_seed"])
for k in ["train", "val", "test"]:
    print(f"  {k}: {len(loaded['split'][k])} images")

SPLIT_HASH = loaded["split_hash"]  # every notebook should carry this into its config.json


Loaded split_hash: 5ba9dc1cc68a1202cd0bb0446e457ab0
Loaded split seed: 42
  train: 3200 images
  val: 400 images
  test: 400 images


### Duplicate-file check (run this once, right after locking the split)

Sanity check for byte-identical images landing in more than one split -- this
can happen if the raw dataset itself contains duplicate files (common in
scraped/aggregated medical imaging datasets). It does NOT catch near-duplicate
video frames (visually similar but not byte-identical) -- only exact copies.


In [9]:
# # import hashlib

# # def file_hash(path):
# #     with open(path, "rb") as f:
# #         return hashlib.md5(f.read()).hexdigest()

# # def hash_split_part(entries):
# #     return {file_hash(p): (p, cls) for p, cls in entries}

# # train_hashed = hash_split_part(loaded["split"]["train"])
# # val_hashed   = hash_split_part(loaded["split"]["val"])
# # test_hashed  = hash_split_part(loaded["split"]["test"])

# # train_test_overlap = set(train_hashed) & set(test_hashed)
# # train_val_overlap  = set(train_hashed) & set(val_hashed)
# # val_test_overlap   = set(val_hashed) & set(test_hashed)

# # print("train/test overlap:", len(train_test_overlap))
# # print("train/val overlap:", len(train_val_overlap))
# # print("val/test overlap:", len(val_test_overlap))

# # if train_test_overlap or train_val_overlap or val_test_overlap:
# #     print("\nDuplicate files found -- details:")
# #     for h in (train_test_overlap | train_val_overlap | val_test_overlap):
# #         for name, d in [("train", train_hashed), ("val", val_hashed), ("test", test_hashed)]:
# #             if h in d:
# #                 print(f"  [{name}] {d[h]}")
# #         print()
# import hashlib, shutil, os

# # Reuse a local copy so hashing reads from fast local disk, not Drive
# LOCAL_DATASET_DIR = "/content/gastro_local_copy"
# if not os.path.exists(LOCAL_DATASET_DIR):
#     print("Copying dataset locally for fast hashing (one-time)...")
#     shutil.copytree(RAW_DATASET_DIR, LOCAL_DATASET_DIR)
#     print("Done.")

# def to_local(path):
#     return path.replace(RAW_DATASET_DIR, LOCAL_DATASET_DIR)

# def file_hash(path):
#     with open(path, "rb") as f:
#         return hashlib.md5(f.read()).hexdigest()

# def hash_split_part(entries, label):
#     result = {}
#     for i, (p, cls) in enumerate(entries):
#         result[file_hash(to_local(p))] = (p, cls)   # store ORIGINAL Drive path, hash from LOCAL copy
#         if (i + 1) % 500 == 0:
#             print(f"  [{label}] hashed {i+1}/{len(entries)}")
#     return result

# train_hashed = hash_split_part(loaded["split"]["train"], "train")
# val_hashed   = hash_split_part(loaded["split"]["val"], "val")
# test_hashed  = hash_split_part(loaded["split"]["test"], "test")

# train_test_overlap = set(train_hashed) & set(test_hashed)
# train_val_overlap  = set(train_hashed) & set(val_hashed)
# val_test_overlap   = set(val_hashed) & set(test_hashed)

# print("train/test overlap:", len(train_test_overlap))
# print("train/val overlap:", len(train_val_overlap))
# print("val/test overlap:", len(val_test_overlap))

# if train_test_overlap or train_val_overlap or val_test_overlap:
#     print("\nDuplicate files found -- details:")
#     for h in (train_test_overlap | train_val_overlap | val_test_overlap):
#         for name, d in [("train", train_hashed), ("val", val_hashed), ("test", test_hashed)]:
#             if h in d:
#                 print(f"  [{name}] {d[h]}")
#         print()import hashlib, shutil, os

import hashlib, shutil, os

LOCAL_DATASET_DIR = "/content/gastro_local_copy"

if not os.path.exists(LOCAL_DATASET_DIR):
    os.makedirs(LOCAL_DATASET_DIR)
    for cls in CLASS_NAMES:
        src_dir = os.path.join(RAW_DATASET_DIR, cls)
        dst_dir = os.path.join(LOCAL_DATASET_DIR, cls)
        os.makedirs(dst_dir, exist_ok=True)
        files = os.listdir(src_dir)
        print(f"Copying class '{cls}': {len(files)} files")
        for i, fname in enumerate(files):
            shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
            if (i + 1) % 200 == 0:
                print(f"  [{cls}] copied {i+1}/{len(files)}")
    print("Local copy complete.\n")
else:
    print("Local copy already exists this session, skipping copy.\n")

def to_local(path):
    return path.replace(RAW_DATASET_DIR, LOCAL_DATASET_DIR)

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def hash_split_part(entries, label):
    result = {}
    for i, (p, cls) in enumerate(entries):
        result[file_hash(to_local(p))] = (p, cls)   # store ORIGINAL Drive path, hash from LOCAL copy
        if (i + 1) % 500 == 0:
            print(f"  [{label}] hashed {i+1}/{len(entries)}")
    print(f"  [{label}] done, {len(entries)} total")
    return result

print("Hashing train...")
train_hashed = hash_split_part(loaded["split"]["train"], "train")
print("Hashing val...")
val_hashed   = hash_split_part(loaded["split"]["val"], "val")
print("Hashing test...")
test_hashed  = hash_split_part(loaded["split"]["test"], "test")

train_test_overlap = set(train_hashed) & set(test_hashed)
train_val_overlap  = set(train_hashed) & set(val_hashed)
val_test_overlap   = set(val_hashed) & set(test_hashed)

print("\ntrain/test overlap:", len(train_test_overlap))
print("train/val overlap:", len(train_val_overlap))
print("val/test overlap:", len(val_test_overlap))

if train_test_overlap or train_val_overlap or val_test_overlap:
    print("\nDuplicate files found -- details:")
    for h in (train_test_overlap | train_val_overlap | val_test_overlap):
        for name, d in [("train", train_hashed), ("val", val_hashed), ("test", test_hashed)]:
            if h in d:
                print(f"  [{name}] {d[h]}")
        print()

Copying class 'Diverticulosis': 1000 files
  [Diverticulosis] copied 200/1000
  [Diverticulosis] copied 400/1000
  [Diverticulosis] copied 600/1000
  [Diverticulosis] copied 800/1000
  [Diverticulosis] copied 1000/1000
Copying class 'Neoplasm': 1000 files
  [Neoplasm] copied 200/1000
  [Neoplasm] copied 400/1000
  [Neoplasm] copied 600/1000
  [Neoplasm] copied 800/1000
  [Neoplasm] copied 1000/1000
Copying class 'Peritonitis': 1000 files
  [Peritonitis] copied 200/1000
  [Peritonitis] copied 400/1000
  [Peritonitis] copied 600/1000
  [Peritonitis] copied 800/1000
  [Peritonitis] copied 1000/1000
Copying class 'Ureters': 1000 files
  [Ureters] copied 200/1000
  [Ureters] copied 400/1000
  [Ureters] copied 600/1000
  [Ureters] copied 800/1000
  [Ureters] copied 1000/1000
Local copy complete.

Hashing train...
  [train] hashed 500/3200
  [train] hashed 1000/3200
  [train] hashed 1500/3200
  [train] hashed 2000/3200
  [train] hashed 2500/3200
  [train] hashed 3000/3200
  [train] done, 3200

### If duplicates were found: generate a v2 split (only run this if Cell 6b found overlap)

This does NOT overwrite `dataset_split.json`. It writes a separate
`dataset_split_v2.json` with the duplicate(s) removed from every split
except one (kept in test, since test-set composition matters most and is
the smallest/least disruptive to shrink from). Every existing `results.json`
trained against the v1 split stays valid as a v1 result -- this just gives
you a clean split to re-run against going forward, and both split files are
distinguishable by their `split_hash`.


In [10]:
def build_v2_split(v1_split, hashes_to_dedupe):
    '''
    Removes each duplicated file from every split part EXCEPT test.
    hashes_to_dedupe: iterable of md5 hashes that appear in >1 split.
    '''
    v2 = {"train": [], "val": [], "test": []}
    seen_hashes_kept = set()

    # Process test first so duplicates are kept there.
    for part in ["test", "val", "train"]:
        for path, cls in v1_split[part]:
            h = file_hash(path)
            if h in hashes_to_dedupe and h in seen_hashes_kept:
                continue  # drop this duplicate occurrence, already kept once (in test)
            v2[part].append((path, cls))
            if h in hashes_to_dedupe:
                seen_hashes_kept.add(h)
    return v2


all_overlap_hashes = train_test_overlap | train_val_overlap | val_test_overlap

if not all_overlap_hashes:
    print("No overlap found in Cell 6b -- nothing to fix, dataset_split.json is clean as-is.")
else:
    v2_split = build_v2_split(loaded["split"], all_overlap_hashes)

    v2_total = sum(len(v2_split[k]) for k in v2_split)
    print(f"v1 total: {sum(len(loaded['split'][k]) for k in loaded['split'])}")
    print(f"v2 total: {v2_total}  (should be v1 total minus {len(all_overlap_hashes)})")
    for k in ["train", "val", "test"]:
        print(f"  {k}: {len(v2_split[k])} images")

    v2_hash = _split_hash(v2_split)
    v2_path = os.path.join(EXPERIMENTS_ROOT, "dataset_split_v2.json")

    v2_payload = {
        "split_seed": SPLIT_SEED,
        "split_ratios": SPLIT_RATIOS,
        "class_names": CLASS_NAMES,
        "split_hash": v2_hash,
        "derived_from": SPLIT_HASH,
        "fix_note": f"Removed {len(all_overlap_hashes)} duplicate file(s) that appeared in multiple splits in v1; duplicates kept in test, removed from train/val.",
        "generated_by_account": ACCOUNT_TAG,
        "generated_by_notebook": NOTEBOOK_NAME,
        "split": v2_split,
    }
    with open(v2_path, "w") as f:
        json.dump(v2_payload, f, indent=2)

    print("\ndataset_split_v2.json written to:", v2_path)
    print("v2 split_hash:", v2_hash)
    print("\ndataset_split.json (v1) is left untouched -- any results already trained")
    print("against it remain valid v1 results. Point new training runs at v2 going")
    print("forward by loading dataset_split_v2.json instead in each notebook's Cell 4.")


v1 total: 4000
v2 total: 3998  (should be v1 total minus 2)
  train: 3198 images
  val: 400 images
  test: 400 images

dataset_split_v2.json written to: /content/drive/MyDrive/gastronet_experiments/dataset_split_v2.json
v2 split_hash: d6e80caa29bff18856ae93ea635b4650

dataset_split.json (v1) is left untouched -- any results already trained
against it remain valid v1 results. Point new training runs at v2 going
forward by loading dataset_split_v2.json instead in each notebook's Cell 4.


In [11]:
checkpoint_utils_code = '''
"""
checkpoint_utils.py — SHARED module, imported by every GastroNet notebook.

Do not copy/paste this into individual notebooks and do not maintain a second
copy. If a change is needed, edit it via NB0 and every notebook that does
`sys.path.append(EXPERIMENTS_ROOT); import checkpoint_utils` will pick it up
on next import (restart runtime if a notebook already imported the old
version in this session).
"""

import os
import json
import time
import shutil
import torch


def get_experiment_dir(experiments_root, model_family, seed):
    """
    Canonical, ONLY path scheme for any experiment. model_family must be one
    of: cnn_only, vit_only, hybrid_concat, hybrid_crossattn.
    """
    path = os.path.join(experiments_root, model_family, f"seed_{seed}")
    os.makedirs(path, exist_ok=True)
    return path


def save_latest(exp_dir, epoch, model, optimizer, scheduler, scaler,
                 best_val_acc, history):
    """
    Full resumable state. Call this every epoch. This file is TEMPORARY
    infrastructure -- it exists only to survive a Colab disconnect mid-run
    and should be removed by finalize_experiment() once the run is done.
    """
    state = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "best_val_acc": best_val_acc,
        "history": history,
    }
    tmp_path = os.path.join(exp_dir, "latest.pt.tmp")
    final_path = os.path.join(exp_dir, "latest.pt")
    torch.save(state, tmp_path)
    os.replace(tmp_path, final_path)   # atomic-ish, avoids a half-written latest.pt


def load_latest(exp_dir, map_location="cpu"):
    """Returns the dict from save_latest(), or None if no checkpoint exists yet."""
    path = os.path.join(exp_dir, "latest.pt")
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location)


def save_best(exp_dir, model, epoch, best_val_acc, config):
    """
    Weights + metadata only. This IS a permanent deliverable -- used for
    final evaluation and for the paper's reported numbers. Never put
    optimizer/scheduler state in here.
    """
    state = {
        "model_state_dict": model.state_dict(),
        "epoch": epoch,
        "best_val_acc": best_val_acc,
        "config": config,
    }
    path = os.path.join(exp_dir, "best.pt")
    tmp_path = path + ".tmp"
    torch.save(state, tmp_path)
    os.replace(tmp_path, path)


def load_best(exp_dir, map_location="cpu"):
    path = os.path.join(exp_dir, "best.pt")
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location)


def save_config(exp_dir, config: dict):
    with open(os.path.join(exp_dir, "config.json"), "w") as f:
        json.dump(config, f, indent=2)


def save_history(exp_dir, history: dict):
    with open(os.path.join(exp_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)


def save_results(exp_dir, results: dict):
    with open(os.path.join(exp_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)


def resume_or_start(exp_dir, model, optimizer, scheduler, scaler, map_location="cpu"):
    """
    Standard entrypoint every training notebook should call right after
    building model/optimizer/scheduler/scaler. Returns (start_epoch,
    best_val_acc, history). Loads latest.pt if present, else starts fresh.
    """
    ckpt = load_latest(exp_dir, map_location=map_location)
    if ckpt is None:
        print(f"[{exp_dir}] No latest.pt found -- starting fresh at epoch 0.")
        return 0, 0.0, {"train_loss": [], "val_loss": [], "val_acc": []}

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt.get("scheduler_state") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if scaler is not None and ckpt.get("scaler_state") is not None:
        scaler.load_state_dict(ckpt["scaler_state"])

    start_epoch = ckpt["epoch"] + 1
    print(f"[{exp_dir}] Resuming from epoch {start_epoch} "
          f"(best_val_acc so far: {ckpt['best_val_acc']:.4f})")
    return start_epoch, ckpt["best_val_acc"], ckpt["history"]


def finalize_experiment(exp_dir, require_best=True, require_results=True):
    """
    Deletes latest.pt -- ONLY call this after you have manually confirmed
    best.pt and results.json are present and correct. This is deliberately
    NOT automatic at the end of a training loop; call it yourself, once,
    when you are sure.
    """
    best_path = os.path.join(exp_dir, "best.pt")
    results_path = os.path.join(exp_dir, "results.json")
    latest_path = os.path.join(exp_dir, "latest.pt")

    if require_best and not os.path.exists(best_path):
        raise RuntimeError(f"Refusing to finalize: {best_path} does not exist yet.")
    if require_results and not os.path.exists(results_path):
        raise RuntimeError(f"Refusing to finalize: {results_path} does not exist yet.")

    if os.path.exists(latest_path):
        os.remove(latest_path)
        print(f"Finalized {exp_dir}: removed latest.pt (best.pt + results.json verified present).")
    else:
        print(f"Finalized {exp_dir}: latest.pt was already absent, nothing to remove.")


def assert_split_hash_matches(config: dict, expected_split_hash: str):
    """Call this before evaluating/reporting any result, in any notebook,
    on any account. This is the cross-account tripwire for Fault 8."""
    got = config.get("split_hash")
    if got != expected_split_hash:
        raise RuntimeError(
            f"SPLIT HASH MISMATCH: this config was built against split_hash={got}, "
            f"but the currently loaded dataset_split.json has hash={expected_split_hash}. "
            f"Do not evaluate -- you are not comparing on the same data."
        )


# ---------------------------------------------------------------
# experiments_manifest.json helpers -- the cross-account log
# ---------------------------------------------------------------

def _load_manifest(manifest_path):
    if not os.path.exists(manifest_path):
        return {"runs": []}
    with open(manifest_path) as f:
        return json.load(f)


def _save_manifest(manifest_path, manifest):
    tmp_path = manifest_path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(manifest, f, indent=2)
    os.replace(tmp_path, manifest_path)


def log_run_event(manifest_path, model_family, seed, account_tag, notebook_name,
                   status, best_val_acc=None, drive_path=None, note=None):
    """
    status should be one of: "started", "resumed", "completed", "interrupted".
    Appends a new entry -- never overwrites history, so the manifest is a
    full timeline across every account and every notebook run.
    """
    manifest = _load_manifest(manifest_path)
    manifest["runs"].append({
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "model_family": model_family,
        "seed": seed,
        "account": account_tag,
        "notebook": notebook_name,
        "status": status,
        "best_val_acc": best_val_acc,
        "drive_path": drive_path,
        "note": note,
    })
    _save_manifest(manifest_path, manifest)


def manifest_summary(manifest_path):
    """Quick glance at the latest status per (model_family, seed)."""
    manifest = _load_manifest(manifest_path)
    latest_by_key = {}
    for run in manifest["runs"]:
        key = (run["model_family"], run["seed"])
        latest_by_key[key] = run   # runs are appended in order, so last wins
    for (model_family, seed), run in sorted(latest_by_key.items()):
        print(f"{model_family:18s} seed_{seed:<6} -> {run['status']:12s} "
              f"acc={run['best_val_acc']} account={run['account']} "
              f"notebook={run['notebook']} at {run['timestamp']}")
'''

with open(CHECKPOINT_UTILS_PATH, "w") as f:
    f.write(checkpoint_utils_code)

print("checkpoint_utils.py written to:", CHECKPOINT_UTILS_PATH)


checkpoint_utils.py written to: /content/drive/MyDrive/gastronet_experiments/checkpoint_utils.py


In [12]:
import sys
sys.path.insert(0, EXPERIMENTS_ROOT)
import checkpoint_utils as cku   # the module we just wrote to Drive

if not os.path.exists(MANIFEST_JSON_PATH):
    with open(MANIFEST_JSON_PATH, "w") as f:
        json.dump({"runs": []}, f, indent=2)
    print("experiments_manifest.json initialized at:", MANIFEST_JSON_PATH)
else:
    print("experiments_manifest.json already exists, leaving it as-is:", MANIFEST_JSON_PATH)

cku.log_run_event(
    MANIFEST_JSON_PATH,
    model_family="_setup",
    seed=0,
    account_tag=ACCOUNT_TAG,
    notebook_name=NOTEBOOK_NAME,
    status="completed",
    note="NB0 setup and split lock completed.",
)
print("\nManifest summary so far:")
cku.manifest_summary(MANIFEST_JSON_PATH)


experiments_manifest.json already exists, leaving it as-is: /content/drive/MyDrive/gastronet_experiments/experiments_manifest.json

Manifest summary so far:
_setup             seed_0      -> completed    acc=None account=acct_A notebook=NB0_setup_and_split_lock at 2026-09-07 10:15:46
cnn_only           seed_7      -> completed    acc=0.985 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:56:38
cnn_only           seed_42     -> resumed      acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:26:57
cnn_only           seed_123    -> completed    acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:42:24


In [14]:
# Proves the whole pipeline works end-to-end with a throwaway dummy model,
# BEFORE you trust it with a real multi-hour training run. Uses a fake
# model_family ("_sanity_check") so it never collides with real experiments.

import torch.nn as nn
import torch
dummy_model = nn.Linear(4, 4)
dummy_optimizer = torch.optim.Adam(dummy_model.parameters(), lr=1e-3)
dummy_scheduler = torch.optim.lr_scheduler.StepLR(dummy_optimizer, step_size=1)
dummy_scaler = torch.cuda.amp.GradScaler(enabled=False)

sanity_dir = cku.get_experiment_dir(EXPERIMENTS_ROOT, "_sanity_check", seed=0)

start_epoch, best_val_acc, history = cku.resume_or_start(
    sanity_dir, dummy_model, dummy_optimizer, dummy_scheduler, dummy_scaler
)
print("start_epoch:", start_epoch, "best_val_acc:", best_val_acc)

cku.save_latest(sanity_dir, epoch=0, model=dummy_model, optimizer=dummy_optimizer,
                 scheduler=dummy_scheduler, scaler=dummy_scaler,
                 best_val_acc=0.5, history={"train_loss": [0.9], "val_loss": [0.8], "val_acc": [0.5]})

dummy_config = {
    "model_family": "_sanity_check",
    "seed": 0,
    "split_hash": SPLIT_HASH,
    "account_tag": ACCOUNT_TAG,
    "notebook_name": NOTEBOOK_NAME,
}
cku.save_config(sanity_dir, dummy_config)
cku.save_best(sanity_dir, dummy_model, epoch=0, best_val_acc=0.5, config=dummy_config)
cku.save_results(sanity_dir, {"test_accuracy": 0.5, "note": "sanity check only, not a real result"})

# Load it all back and verify the split hash check works
loaded_config = json.load(open(os.path.join(sanity_dir, "config.json")))
cku.assert_split_hash_matches(loaded_config, SPLIT_HASH)
print("\nSplit hash check passed.")

reloaded_latest = cku.load_latest(sanity_dir)
reloaded_best = cku.load_best(sanity_dir)
print("latest.pt round-trip OK, epoch =", reloaded_latest["epoch"])
print("best.pt round-trip OK, best_val_acc =", reloaded_best["best_val_acc"])

cku.finalize_experiment(sanity_dir)
print("\nSanity check complete -- checkpoint_utils.py is working correctly end-to-end.")
print("(This _sanity_check folder is harmless to leave, or delete it manually if you prefer.)")


/tmp/ipykernel_1325/2941765384.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  dummy_scaler = torch.cuda.amp.GradScaler(enabled=False)


[/content/drive/MyDrive/gastronet_experiments/_sanity_check/seed_0] No latest.pt found -- starting fresh at epoch 0.
start_epoch: 0 best_val_acc: 0.0

Split hash check passed.
latest.pt round-trip OK, epoch = 0
best.pt round-trip OK, best_val_acc = 0.5
Finalized /content/drive/MyDrive/gastronet_experiments/_sanity_check/seed_0: removed latest.pt (best.pt + results.json verified present).

Sanity check complete -- checkpoint_utils.py is working correctly end-to-end.
(This _sanity_check folder is harmless to leave, or delete it manually if you prefer.)


## NB0 complete. What is now permanently locked on this account's Drive:

- `gastronet_experiments/dataset_split.json` — the original locked split (v1), deterministic, hash-verified.
- `gastronet_experiments/dataset_split_v2.json` — present only if Cell 6b found duplicate files; a corrected split with the duplicate(s) removed from train/val, kept in test.
- `gastronet_experiments/checkpoint_utils.py` — the shared module every other notebook imports (never redefine these functions elsewhere).
- `gastronet_experiments/experiments_manifest.json` — the running log of every experiment, across every account.
- Empty top-level folders for `cnn_only/`, `vit_only/`, `hybrid_concat/`, `hybrid_crossattn/`.

### If a v2 split was generated
Any experiments already trained against v1 remain valid v1 results — don't discard them, just don't compare them directly against v2 results. Going forward, point new/re-run training notebooks at `dataset_split_v2.json` instead of `dataset_split.json` (a one-line change to the `SPLIT_JSON_PATH` variable in each notebook's config cell). Re-run any seeds you want a clean v2 comparison for.

### To replicate on a second Google account
1. Share the `gastronet_experiments` Drive folder from this account to the other account (or download+reupload just `dataset_split.json` and `checkpoint_utils.py` if full sharing isn't convenient).
2. In the new account's copy, do **not** re-run Cell 5. Just run Cells 2–4 and 6–8 (mount, config, folder creation, load-and-verify split, import checkpoint_utils, touch the manifest) so that account's local paths and `sys.path` are set up identically.
3. Set `ACCOUNT_TAG` in Cell 3 to a distinct value (e.g. `acct_B`) before running anything there — every checkpoint and manifest entry created from that account will carry this tag.

### Standing rules from here on (do not relitigate these)
- Every other notebook starts with: mount Drive → `sys.path.append(EXPERIMENTS_ROOT)` → `import checkpoint_utils as cku` → load `dataset_split.json` → assert its hash matches what's expected.
- Every training run calls `cku.resume_or_start(...)` immediately after building model/optimizer/scheduler/scaler — never a bespoke resume check.
- Every run logs to the manifest at start and at completion/interruption via `cku.log_run_event(...)`.
- `finalize_experiment()` is called manually, once, per experiment, only after you've confirmed `best.pt` and `results.json` are correct — never automatically at the end of a training loop.
